In [1]:
import random, numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

OUT = Path("../data/processed")
LABELS = ["Exact", "Hybrid", "Semantic"]            # index 0 = Exact -> alpha = probs[:, 0]
l2i = {l: i for i, l in enumerate(LABELS)}

tr, va, te = [pd.read_csv(OUT / f"classifier_{n}.csv") for n in ["train", "val", "test"]]
for n, d in zip(["train", "val", "test"], [tr, va, te]):
    print(n, len(d), d["intent"].value_counts().to_dict())

order = pd.read_parquet(OUT / "eval_query_order.parquet")
ids = set(tr.query_id) | set(va.query_id) | set(te.query_id)
print("\nid overlap with dev/test retrieval queries:", len(ids & set(order.query_id)))
norm = lambda s: s.str.lower().str.strip()
print("duplicate text train-val:", len(set(norm(tr["query"])) & set(norm(va["query"]))),
      "| train-test:", len(set(norm(tr["query"])) & set(norm(te["query"]))))

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
Xtr, Xva = vec.fit_transform(tr["query"]), vec.transform(va["query"])
lr = LogisticRegression(max_iter=2000, C=5).fit(Xtr, tr["intent"])
pv = lr.predict(Xva)
print(f"\nTF-IDF+LogReg baseline (val): acc {accuracy_score(va['intent'], pv):.3f} | macro-F1 {f1_score(va['intent'], pv, average='macro'):.3f}")

train 386 {'Hybrid': 151, 'Semantic': 143, 'Exact': 92}
val 83 {'Hybrid': 33, 'Semantic': 30, 'Exact': 20}
test 83 {'Hybrid': 32, 'Semantic': 31, 'Exact': 20}

id overlap with dev/test retrieval queries: 0
duplicate text train-val: 0 | train-test: 0

TF-IDF+LogReg baseline (val): acc 0.651 | macro-F1 0.634


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset
import copy

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def make_ds(df, with_labels=True):
    e = tok(df["query"].tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="pt")
    t = [e["input_ids"], e["attention_mask"]]
    if with_labels:
        t.append(torch.tensor(df["intent"].map(l2i).values))
    return TensorDataset(*t)

@torch.no_grad()
def predict_proba(model, df):
    model.eval(); out = []
    for b in DataLoader(make_ds(df, False), batch_size=64):
        logits = model(input_ids=b[0].to(device), attention_mask=b[1].to(device)).logits
        out.append(torch.softmax(logits, -1).cpu().numpy())
    return np.vstack(out)

def train(seed=42, epochs=8, lr=3e-5, bs=16):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3).to(device)
    dl = DataLoader(make_ds(tr), batch_size=bs, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * epochs * len(dl)), epochs * len(dl))
    best, hist = (-1, None, 0), []
    for ep in range(1, epochs + 1):
        model.train(); tot = 0
        for ids_, mask, y in dl:
            loss = model(input_ids=ids_.to(device), attention_mask=mask.to(device), labels=y.to(device)).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad(); tot += loss.item()
        p = predict_proba(model, va); pred = p.argmax(1); y = va["intent"].map(l2i).values
        acc, f1 = accuracy_score(y, pred), f1_score(y, pred, average="macro")
        nll = -np.log(p[np.arange(len(y)), y] + 1e-9).mean()
        hist.append((ep, tot / len(dl), acc, f1, nll)); print(f"ep{ep}  train_loss {tot/len(dl):.3f}  val_acc {acc:.3f}  val_macroF1 {f1:.3f}  val_nll {nll:.3f}")
        if f1 > best[0]:
            best = (f1, copy.deepcopy(model.state_dict()), ep)
    model.load_state_dict(best[1]); print(f"\nbest epoch {best[2]} (val macro-F1 {best[0]:.3f})")
    return model, hist

model, hist = train()
model.save_pretrained(OUT / "clf_distilbert"); tok.save_pretrained(OUT / "clf_distilbert")

pv = predict_proba(model, va).argmax(1); yv = va["intent"].map(l2i).values
print("\nVAL report:"); print(classification_report(yv, pv, target_names=LABELS, digits=3))
print(pd.DataFrame(confusion_matrix(yv, pv), index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS]))

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 571.45it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ep1  train_loss 1.065  val_acc 0.530  val_macroF1 0.398  val_nll 1.006
ep2  train_loss 0.849  val_acc 0.723  val_macroF1 0.711  val_nll 0.737
ep3  train_loss 0.528  val_acc 0.795  val_macroF1 0.784  val_nll 0.575
ep4  train_loss 0.275  val_acc 0.771  val_macroF1 0.750  val_nll 0.584
ep5  train_loss 0.151  val_acc 0.771  val_macroF1 0.757  val_nll 0.674
ep6  train_loss 0.080  val_acc 0.807  val_macroF1 0.803  val_nll 0.710
ep7  train_loss 0.060  val_acc 0.771  val_macroF1 0.750  val_nll 0.811
ep8  train_loss 0.040  val_acc 0.771  val_macroF1 0.756  val_nll 0.812

best epoch 6 (val macro-F1 0.803)


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.28s/it]



VAL report:
              precision    recall  f1-score   support

       Exact      0.824     0.700     0.757        20
      Hybrid      0.725     0.879     0.795        33
    Semantic      0.923     0.800     0.857        30

    accuracy                          0.807        83
   macro avg      0.824     0.793     0.803        83
weighted avg      0.820     0.807     0.808        83

               pred_Exact  pred_Hybrid  pred_Semantic
true_Exact             14            6              0
true_Hybrid             2           29              2
true_Semantic           1            5             24
